# పాఠం 05 - ఏజెంటిక్ RAG


## సెటప్

ఈ నోట్బుక్ మైక్రోసాఫ్ట్ ఏజెంట్ ఫ్రేమ్‌వర్క్ ఉపయోగించి ఏజెంటిక్ RAG (రిట్రీవల్-ఆగ్మెంటెడ్ జనరేషన్) నమూనాను ప్రదర్శిస్తుంది.

**ముందస్తు అవసరాలు:**
- `AZURE_SEARCH_SERVICE_ENDPOINT` — మీ Azure AI సెర్చ్ సర్వీస్ ఎండ్‌పాయింట్
- `AZURE_SEARCH_API_KEY` — మీ Azure AI సెర్చ్ API కీ
- పర్యావరణ వేరియబుల్స్ ద్వారా కాన్ఫిగర్ చేయబడిన Azure OpenAI డిప్లాయ్‌మెంట్
- Azure CLI ధృవీకరించబడింది (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Azure AI Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## ఏజెంటిక్ RAG అంటే ఏమిటి?

సాంప్రదాయ RAG ఒక స్థిరమైన పైప్‌లైన్‌ను అనుసరిస్తుంది: మొదట డాక్యుమెంట్లను తీసుకొని, అనంతరం ప్రతిస్పందనను రూపొందిస్తుంది. **ఏజెంటిక్ RAG** ఏజెంట్‌కు **ఎప్పుడు** మరియు **ఎట్లుగా** సమాచారాన్ని తీసుకోవాలని స్వయం నిర్ణయం తీసుకునే స్వాయత్తాన్ని ఇస్తుంది.

ఏజెంటిక్ RAG తో, ఏజెంట్ చేయగలదు:
- ప్రశ్న అడగడానికి ముందుగా తీసుకోకోవలసిన అవసరం ఉందా లేదా **నిర్ణయించు**
- ఏ డేటా మూలం లేదా టూల్‌ని అన్వేషించాలో **ఎంచుకోండి**
- పొందిన ఫలితాలను **మూల్యాంకన చేసి** మొదటి ప్రయత్నం సక్రమం కానప్పుడు మళ్లీ సమాచారాన్ని తీసుకోండి
- అనేక తీసుకునే దశల నుండి సమాచారాన్ని ఒక సुसంగతమైన సమాధానంగా **కలపండి**

ఇది ఏజెంట్‌ను ఒక స్థిరమైన తీసుకో-తర్వాత-రూపొందించే పైప్‌లైన్‌తో కంటే మరింత సరళమైన మరియు ఖచ్చితమైనదిగా చేస్తుంది.


## శోధన సాధనం సృష్టించడం

Agentic RAGలో, బయటి డేటా మూలాలు ఏజెంట్ డిమాండ్ బట్టి పిలవగల **సాధనాలు** (tools)గా ర్యాప్ చేయబడతాయి. ఇది ఏజెంట్ని తీసుకోవలసిన చర్యలలో ఒకటిగా రిట్రీవల్‌ని చూడటానికి సహాయపడుతుంది, తప్పనిసరి దశగా కాకుండా.

కింద మనం ఒక ప్రయాణ జ్ఞాన ఆధారాన్ని నిర్వచించి, దాని గమ్యస్థాన సమాచారాన్ని చూసుకోవడానికి ఏజెంట్ పిలవగల సాధనంగా అందిస్తున్నాము.


In [ ]:
TRAVEL_KNOWLEDGE_BASE = {
    "Barcelona": "Barcelona is Spain's cosmopolitan capital of Catalonia. Best visited Mar-May or Sep-Nov. Known for Gaudí architecture, La Rambla, beaches. Average daily cost: $150-200.",
    "Tokyo": "Tokyo is Japan's capital, mixing ultramodern with traditional. Best visited Mar-Apr (cherry blossoms) or Oct-Nov. Known for Shibuya, temples, sushi. Average daily cost: $200-250.",
    "Paris": "Paris is France's capital and a global center for art, fashion, and culture. Best visited Apr-Jun or Sep-Oct. Known for Eiffel Tower, Louvre, cuisine. Average daily cost: $180-250.",
    "Cape Town": "Cape Town sits on South Africa's southwest tip. Best visited Nov-Mar. Known for Table Mountain, wine regions, wildlife. Average daily cost: $100-150.",
}


@tool(approval_mode="never_require")
def search_travel_knowledge(
    query: Annotated[str, "The search query about a travel destination"]
) -> str:
    """Search the travel knowledge base for destination information."""
    results = []
    for destination, info in TRAVEL_KNOWLEDGE_BASE.items():
        if query.lower() in destination.lower() or any(
            word in info.lower() for word in query.lower().split()
        ):
            results.append(f"**{destination}**: {info}")
    return (
        "\n\n".join(results)
        if results
        else "No matching destinations found in the knowledge base."
    )

## RAG ఏజెంట్ నిర్మించడం

ఇప్పుడు మనం ఒక ఏజెంట్‌ను సృష్టించబోతున్నాము, ఇది **ఎప్పుడూ సమాధానం చెప్పే ముందు సమాచారం సేకరించాలి** అని సూచించబడింది. ఏజెంట్ తన స్వంత శిక్షణ డేటాపైఆధారపడడం కాకుండా, దాని స్పందనలను జ్ఞాన ఆధారంలో బేస్ చేసుకోవడానికి `search_travel_knowledge` టూల్‌ని ఉపయోగిస్తుంది.


In [ ]:
agent = client.as_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGAgent",
    instructions="""You are a knowledgeable travel advisor. Before answering questions about destinations:
1. ALWAYS search the travel knowledge base first
2. Base your answers on retrieved information
3. If information is not in the knowledge base, say so clearly
4. Provide specific details like costs, best seasons, and highlights.""",
)

response = await agent.run(
    "I'm interested in visiting somewhere with great architecture. What destinations would you recommend?",
    )
print(response)

## పునఃప్రాప్తి — మేకర్-చెక్కర్ నమూనా

Agentic RAG ప్రధాన ప్రయోజనం **పునఃప్రాప్తి**. ఏజెంట్ తన ప్రారంభ కనుగొనింపులు సరిచూసుకోవడానికి, మెరుగుపరచడానికి లేదా విస్తరించడానికి అనేక రౌండ్ల శోధన చేయవచ్చు — ఇది "మేకర్-చేకర్" వర్క్‌ఫ్లో కి సారూప్యంగా ఉంటుంది:

1. **మేకర్ దశ**: ఏజెంట్ ప్రారంభ సమాచారం ప్రాప్తి చేసి ఒక ప్రతిస్పందనను రూపకల్పన చేస్తుంది.
2. **చెక్కర్ దశ**: ఏజెంట్ వివరాలను ధృవీకరించడానికి లేదా ఖాళీలను నింపడానికి అదనపు ప్రాప్తులు చేస్తుంది.

క్రింది భాగంలో, ఏజెంట్ ను అనేక గమ్యస్థానాలను సరిపోల్చాల్సిన ప్రశ్న అడిగారు, దీనివల్ల అది ఒకাধিক సార్లు శోధించేందుకు ప్రేరేపింపబడుతుంది.


In [ ]:
checker_agent = client.as_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGCheckerAgent",
    instructions="""You are a meticulous travel advisor who double-checks recommendations.
When answering travel questions:
1. Search for relevant destinations first
2. For each destination found, search again with the destination name to get full details
3. Compare the options using verified information
4. Present a final recommendation with specific costs, best travel times, and highlights
5. If any detail seems incomplete, search once more to confirm before responding.""",
)

response = await checker_agent.run(
    "I have a $175/day budget and want to travel in April. Which destinations fit my budget and timing?",
    )
print(response)

## సారాంశం

ఈ పాఠంలో మీరు Microsoft Agent Framework ను ఉపయోగించి **Agentic RAG** సిస్టమ్‌ను ఎలా నిర్మించాలో నేర్చుకున్నారు:

- **Agentic RAG** ఏజెంట్లకు స్వయంచాలకంగా సమాచారం రాబట్టుకోవడానికి ఎప్పుడు నిర్ణయించుకోవచ్చును, దీని వల్ల రిట్రీవల్ స్థిరమైనదిగా కాక, డైనమిక్‌గా మారుతుంది.
- **టూల్స్‌ను డేటా మూలాలుగా**: బాహ్య జ్ఞానాధారాలు (Azure AI Search లాంటి) ఏజెంట్ పిలవగల టూల్స్‌గా ముడిపడతాయి.
- **పునరావృత రిట్రీవల్**: తయారీదారు-తपासకుడు నమూనా ఏజెంట్‌కు పలు రిట్రీవల్ రౌండ్లను — శోధించడం, ధృవీకరించడం, మరియు మెరుగుపరచడం — నిర్వహించేందుకు సహాయపడుతుంది, తుది సమాధానం ఇచ్చే ముందు.

ప్రొడక్షన్ లో, మీరు ఇన్-మెమరీ `TRAVEL_KNOWLEDGE_BASE` ను పెద్ద స్థాయిలో ప్రయాణ దస్త్ర రిట్రీవల్ నిర్వహించడానికి అసలు Azure AI Search సూచికతో మార్చేవారు.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**అస్వీకరణ**:
ఈ పత్రం AI అనువాద సేవ [Co-op Translator](https://github.com/Azure/co-op-translator) ఉపయోగించి అనువదించబడింది. మేము ఖచ్చితత్వానికి ప్రయత్నిస్తున్నప్పటికీ, ఆటోమేటెడ్ అనువాదాలు తప్పులు లేదా అసమగ్రతలను కలిగి ఉండవచ్చు. దాని స్వదేశ భాషలో ఉన్న అసలు పత్రాన్ని అధికారం కలిగిన మూలంగా పరిగణించాలి. కీలకమైన సమాచారం కోసం, ప్రొఫెషనల్ మానవ అనువాదాన్ని సిఫారసు చేస్తాము. ఈ అనువాదం ఉపయోగం వల్ల కలిగే ఏవైనా అపార్థాలు లేదా తప్పుదారులు కోసం మేము బాధ్యత వహించము.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
